In [1]:
import polars as pl
import polars.selectors as cs
from glob import glob
from toolz import pipe
from functools import reduce
from operator import add, mul

In [2]:
# Remove the bad file (See below)
!rm ./data/uber/uber-bad.csv

'rm' is not recognized as an internal or external command,
operable program or batch file.


# Column exploration using `polars` tables.

In this lecture, we will explore how to use `polars` tables to explore the columns across many files. This will help us find and fix problems with the naming and order of columns across the files.

**Basic procedure:** We want to make a column summary table that shows which columns are present in each file. We will do this by:
1. Use `glob` to find all files matching a pattern.
1. Read in each file as a list of `polars` tables.
2. Stack all columns and aggregate to find unique columns and their counts.
3. Create an columns containing the literal value of `1`.
4. Use a reduction to join all the tables together on the column names.
5. Replace missing values with `0`.
6. Explore the resulting table to find problems, e.g. columns that are not in all files, columns with different capitalization, etc.

### Example - Exploring the columns in Uber data files

#### Step 1: Use `glob` to find all files matching a pattern

In [3]:
(uber_paths :=
 glob('./data/uber/*.csv')
)

['./data/uber\\uber-raw-data-apr14-sample.csv',
 './data/uber\\uber-raw-data-aug14-sample.csv',
 './data/uber\\uber-raw-data-jul14-sample.csv',
 './data/uber\\uber-raw-data-jun14-sample.csv',
 './data/uber\\uber-raw-data-may14-sample.csv',
 './data/uber\\uber-raw-data-sep14-sample.csv']

#### Step 2: Read and process each file

**Procedure:**
1. Read one row from each file.
2. Add a new column containing the file name.
3. Use `unpivot` to stack all columns except the file name.
4. Drop the values column.
5. Add a new column containing the literal value of `1`.
6. Use `pivot` to unstack the table so that each file is a column.

In [4]:
(uber_tables :=
 [pl.read_csv(p)
    .head(1)
    # .with_columns(file = pl.lit(p.replace('\\','/').split('/')[-1]))
    # .unpivot(index='file', variable_name='Column')
    # .drop('value')
    # .with_columns(pl.lit(1).alias('ones'))
    # .pivot(index = 'Column', columns='file', values='ones')
  for p in uber_paths
  ]
)

[shape: (1, 4)
 ┌────────────────────┬─────────┬──────────┬────────┐
 │ Date/Time          ┆ Lat     ┆ Lon      ┆ Base   │
 │ ---                ┆ ---     ┆ ---      ┆ ---    │
 │ str                ┆ f64     ┆ f64      ┆ str    │
 ╞════════════════════╪═════════╪══════════╪════════╡
 │ 4/18/2014 21:38:00 ┆ 40.7359 ┆ -73.9852 ┆ B02682 │
 └────────────────────┴─────────┴──────────┴────────┘,
 shape: (1, 4)
 ┌────────────────────┬─────────┬──────────┬────────┐
 │ Date/Time          ┆ Lat     ┆ Lon      ┆ Base   │
 │ ---                ┆ ---     ┆ ---      ┆ ---    │
 │ str                ┆ f64     ┆ f64      ┆ str    │
 ╞════════════════════╪═════════╪══════════╪════════╡
 │ 8/12/2014 19:19:00 ┆ 40.7062 ┆ -74.0145 ┆ B02598 │
 └────────────────────┴─────────┴──────────┴────────┘,
 shape: (1, 4)
 ┌────────────────────┬────────┬──────────┬────────┐
 │ Date/Time          ┆ Lat    ┆ Lon      ┆ Base   │
 │ ---                ┆ ---    ┆ ---      ┆ ---    │
 │ str                ┆ f64    ┆ f64  

#### Step 3: Combine all the processed tables into a single table

In [5]:
df1 = pl.DataFrame({'Column': ['a', 'b'],
                    'file1.csv':2*[1]})
df2 = pl.DataFrame({'Column': ['a', 'c'],
                    'file2.csv':2*[1]})

In [6]:
(df1.join(df2, 
          on='Column', 
          how = 'outer', 
          suffix = '_right',
          )
    .with_columns(Column = pl.coalesce('Column', 'Column_right'))
    .drop('Column_right')
)

C:\Users\kh6102sj\AppData\Local\Temp\ipykernel_9908\297218877.py:1: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  (df1.join(df2,


Column,file1.csv,file2.csv
str,i64,i64
"""a""",1,1
"""c""",null,1
"""b""",1,null


In [7]:
join_next = lambda df1, df2: (df1.join(df2, 
                                       on='Column', 
                                       how = 'outer', 
                                       suffix = '_right')
                                 .with_columns(Column = pl.coalesce('Column', 'Column_right'))
                                 .drop('Column_right')
                             )

(combined_tables :=
 reduce(join_next, uber_tables)
 .fill_null(0)
 .sort('Column') # Should help find similar names/spellings/cases
)

C:\Users\kh6102sj\AppData\Local\Temp\ipykernel_9908\3979226099.py:1: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  join_next = lambda df1, df2: (df1.join(df2,


ColumnNotFoundError: unable to find column "Column"; valid columns: ["Date/Time", "Lat", "Lon", "Base"]

#### Step 5: Explore the combined table

In [8]:
(combined_tables :=
 combined_tables
 .with_columns(all = pl.reduce(mul, cs.starts_with('uber')),       # Multiple 1/0 columns <==> AND
               count = pl.reduce(add, cs.starts_with('uber')),
               )
)

NameError: name 'combined_tables' is not defined

### Putting it all together

In [9]:
join_next = lambda df1, df2: (df1.join(df2, on='Column', how = 'full', suffix = '_right')
                                 .with_columns(Column = pl.coalesce('Column', 'Column_right'))
                                 .drop('Column_right')
                             )


(uber_column_summary :=
 pipe(glob('./data/uber/*.csv'),
     lambda L: [pl.read_csv(p)
                  .with_columns(file = pl.lit(p.replace('\\','/').split('/')[-1]))
                  .head(1) 
                  .unpivot(index='file', 
                           variable_name='Column')
                  .drop('value')
                  .with_columns(pl.lit(1).alias('ones'))
                  .pivot(index = 'Column', 
                         on='file', 
                         values='ones') 
                for p in L 
                ],
     lambda L: reduce(join_next, L).sort('Column').fill_null(0),
     lambda df: df.with_columns(all = pl.reduce(mul, cs.starts_with('uber')), 
                                count = pl.reduce(add, cs.starts_with('uber'))
                               ),
     )
)

Column,uber-raw-data-apr14-sample.csv,uber-raw-data-aug14-sample.csv,uber-raw-data-jul14-sample.csv,uber-raw-data-jun14-sample.csv,uber-raw-data-may14-sample.csv,uber-raw-data-sep14-sample.csv,all,count
str,i32,i32,i32,i32,i32,i32,i32,i32
"""Base""",1,1,1,1,1,1,1,6
"""Date/Time""",1,1,1,1,1,1,1,6
"""Lat""",1,1,1,1,1,1,1,6
"""Lon""",1,1,1,1,1,1,1,6


#### Look for missing columns

In [10]:
(uber_column_summary
 .filter(pl.col('all') == 0)
)

Column,uber-raw-data-apr14-sample.csv,uber-raw-data-aug14-sample.csv,uber-raw-data-jul14-sample.csv,uber-raw-data-jun14-sample.csv,uber-raw-data-may14-sample.csv,uber-raw-data-sep14-sample.csv,all,count
str,i32,i32,i32,i32,i32,i32,i32,i32


#### Find columns that are not in all files with a count 

In [ ]:

(uber_column_summary
 .filter(pl.col('count') != pl.col('count').max())
)

Column,uber-raw-data-apr14-sample.csv,uber-raw-data-aug14-sample.csv,uber-raw-data-jul14-sample.csv,uber-raw-data-jun14-sample.csv,uber-raw-data-may14-sample.csv,uber-raw-data-sep14-sample.csv,all,count
str,i32,i32,i32,i32,i32,i32,i32,i32


## What if things go wrong?

Now let's manufacture some problems with the files and see how we can use our column exploration table to find and fix them.

In [12]:
f = uber_paths[0] 
original_columns = (pl.read_csv(f).head(1).columns)

(pl.read_csv(f)
   .rename({c:c.upper() for c in original_columns})
   .write_csv(f'data/uber/uber-bad.csv')
)

#### Redo the column summary table to see what has changed.

In [13]:
join_next = lambda df1, df2: (df1.join(df2, on='Column', how = 'full', suffix = '_right')
                                 .with_columns(Column = pl.coalesce('Column', 'Column_right'))
                                 .drop('Column_right')
                             )


(uber_column_summary :=
 pipe(glob('./data/uber/*.csv'),
     lambda L: [pl.read_csv(p)
                  .with_columns(file = pl.lit(p.split('/')[-1]))
                  .head(1) 
                  .unpivot(index='file', 
                           variable_name='Column')
                  .drop('value')
                  .with_columns(pl.lit(1).alias('ones'))
                  .pivot(index = 'Column', 
                         on='file', 
                         values='ones') 
                for p in L 
                ],
     lambda L: reduce(join_next, L).sort('Column').fill_null(0),
     lambda df: df.with_columns(all = pl.reduce(mul, cs.starts_with('uber')), 
                                count = pl.reduce(add, cs.starts_with('uber'))
                               ),
     )
)

Column,uber\uber-bad.csv,uber\uber-raw-data-apr14-sample.csv,uber\uber-raw-data-aug14-sample.csv,uber\uber-raw-data-jul14-sample.csv,uber\uber-raw-data-jun14-sample.csv,uber\uber-raw-data-may14-sample.csv,uber\uber-raw-data-sep14-sample.csv,all,count
str,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""BASE""",1,0,0,0,0,0,0,0,1
"""Base""",0,1,1,1,1,1,1,0,6
"""DATE/TIME""",1,0,0,0,0,0,0,0,1
"""Date/Time""",0,1,1,1,1,1,1,0,6
"""LAT""",1,0,0,0,0,0,0,0,1
"""LON""",1,0,0,0,0,0,0,0,1
"""Lat""",0,1,1,1,1,1,1,0,6
"""Lon""",0,1,1,1,1,1,1,0,6


#### Look for missing columns

In [14]:
(uber_column_summary
 .filter(pl.col('all') == 0)
)

Column,uber\uber-bad.csv,uber\uber-raw-data-apr14-sample.csv,uber\uber-raw-data-aug14-sample.csv,uber\uber-raw-data-jul14-sample.csv,uber\uber-raw-data-jun14-sample.csv,uber\uber-raw-data-may14-sample.csv,uber\uber-raw-data-sep14-sample.csv,all,count
str,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""BASE""",1,0,0,0,0,0,0,0,1
"""Base""",0,1,1,1,1,1,1,0,6
"""DATE/TIME""",1,0,0,0,0,0,0,0,1
"""Date/Time""",0,1,1,1,1,1,1,0,6
"""LAT""",1,0,0,0,0,0,0,0,1
"""LON""",1,0,0,0,0,0,0,0,1
"""Lat""",0,1,1,1,1,1,1,0,6
"""Lon""",0,1,1,1,1,1,1,0,6


#### Find columns that are not in all files with a count 

In [15]:
(uber_column_summary
 .filter(pl.col('count') != pl.col('count').max())
)

Column,uber\uber-bad.csv,uber\uber-raw-data-apr14-sample.csv,uber\uber-raw-data-aug14-sample.csv,uber\uber-raw-data-jul14-sample.csv,uber\uber-raw-data-jun14-sample.csv,uber\uber-raw-data-may14-sample.csv,uber\uber-raw-data-sep14-sample.csv,all,count
str,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""BASE""",1,0,0,0,0,0,0,0,1
"""DATE/TIME""",1,0,0,0,0,0,0,0,1
"""LAT""",1,0,0,0,0,0,0,0,1
"""LON""",1,0,0,0,0,0,0,0,1


## Fixing problems

**Basic procedure:** To fix problems with column names, we will:
1. Read all original columns into a dict of dict.  The outer dict will have a key mapping to the file name and a value mapping to a dict of column names for that file.  The inner dict will have a key mapping to the original column name and a value mapping to the original column name.  
2. These inner dict are meant to be used to rename columns.  We will use a function to apply the following transformations:
   - Renaming columns
   - Changing the case of columns
3. Additionally, we may need use `select` to reorder columns and add/remove missing columns.
   - Reordering columns
   - Adding/removing missing columns
   - Recasting columns to a specific data type
  

**Note.** This is a manual process that requires knowledge of the data and the desired column names and order, so solutions will vary.

In [16]:
(uber_paths :=
 glob('./data/uber/*.csv')
)

['./data/uber\\uber-bad.csv',
 './data/uber\\uber-raw-data-apr14-sample.csv',
 './data/uber\\uber-raw-data-aug14-sample.csv',
 './data/uber\\uber-raw-data-jul14-sample.csv',
 './data/uber\\uber-raw-data-jun14-sample.csv',
 './data/uber\\uber-raw-data-may14-sample.csv',
 './data/uber\\uber-raw-data-sep14-sample.csv']

In [17]:
(original_columns :=
 {p:{col: col
     for col in 
     pl.read_csv(p).columns
    }
  for p in uber_paths
 }

)


{'./data/uber\\uber-bad.csv': {'DATE/TIME': 'DATE/TIME',
  'LAT': 'LAT',
  'LON': 'LON',
  'BASE': 'BASE'},
 './data/uber\\uber-raw-data-apr14-sample.csv': {'Date/Time': 'Date/Time',
  'Lat': 'Lat',
  'Lon': 'Lon',
  'Base': 'Base'},
 './data/uber\\uber-raw-data-aug14-sample.csv': {'Date/Time': 'Date/Time',
  'Lat': 'Lat',
  'Lon': 'Lon',
  'Base': 'Base'},
 './data/uber\\uber-raw-data-jul14-sample.csv': {'Date/Time': 'Date/Time',
  'Lat': 'Lat',
  'Lon': 'Lon',
  'Base': 'Base'},
 './data/uber\\uber-raw-data-jun14-sample.csv': {'Date/Time': 'Date/Time',
  'Lat': 'Lat',
  'Lon': 'Lon',
  'Base': 'Base'},
 './data/uber\\uber-raw-data-may14-sample.csv': {'Date/Time': 'Date/Time',
  'Lat': 'Lat',
  'Lon': 'Lon',
  'Base': 'Base'},
 './data/uber\\uber-raw-data-sep14-sample.csv': {'Date/Time': 'Date/Time',
  'Lat': 'Lat',
  'Lon': 'Lon',
  'Base': 'Base'}}

In [18]:
(fixed_columns :=
 {**original_columns, 
  './data/uber/uber-bad.csv':{col:col.title() 
                             for col in original_columns['./data/uber/uber-bad.csv']
                             },
}
)

KeyError: './data/uber/uber-bad.csv'

#### Make a type and order specification for the correct columns (Brute force)

In [19]:
(col_and_types := {'Date/Time': pl.String(),
                   'Lat': pl.Float64(),
                   'Lon': pl.Float64(),
                   'Base': pl.String(),
                  }
)


{'Date/Time': String, 'Lat': Float64, 'Lon': Float64, 'Base': String}

#### Make a type and order specification (programmatic)

In [20]:
(example_correct_table := pl.read_csv(uber_paths[0]).head()
)

DATE/TIME,LAT,LON,BASE
str,f64,f64,str
"""4/18/2014 21:38:00""",40.7359,-73.9852,"""B02682"""
"""4/23/2014 15:19:00""",40.7642,-73.9543,"""B02598"""
"""4/10/2014 7:15:00""",40.7138,-74.0103,"""B02598"""
"""4/11/2014 15:23:00""",40.7847,-73.9698,"""B02682"""
"""4/7/2014 17:26:00""",40.646,-73.7767,"""B02598"""


In [21]:
(str_columns := example_correct_table.select(cs.string()).columns
)

['DATE/TIME', 'BASE']

In [22]:
(float_columns := example_correct_table.select(cs.float()).columns
)

['LAT', 'LON']

In [23]:
(col_and_types := {c: pl.String() for c in str_columns
                  } |    # Merge operator
                  {c:pl.Float64() for c in float_columns
                  }
)

{'DATE/TIME': String, 'BASE': String, 'LAT': Float64, 'LON': Float64}

In [24]:
(uber_combined :=
 pl.concat([pl.read_csv(p)
              .rename(col_rename)
              .select([pl.col(c).cast(t) for c, t in col_and_types.items()])     # Reorder, remove, or recast columns as needed.
            for p, col_rename in fixed_columns.items()
           ]

))

NameError: name 'fixed_columns' is not defined

In [25]:
# Remove the bad file (See below)
!rm ./data/uber/uber-bad.csv

'rm' is not recognized as an internal or external command,
operable program or batch file.


## <font color = "red"> Exercise 4.3 </font>

### Task: Explore the column names in the City Bike data files

The data folder contains a set of City Bike data files. Explore the column names in these files to find and fix any problems.  Provide a summary of the problems you found and how you fixed them.

In [26]:
# Your code here
(bike_paths :=
 glob('./data/city_bike/*.csv')
)

['./data/city_bike\\JC-201604-citibike-tripdata.csv',
 './data/city_bike\\JC-201605-citibike-tripdata.csv',
 './data/city_bike\\JC-201606-citibike-tripdata.csv',
 './data/city_bike\\JC-201607-citibike-tripdata.csv',
 './data/city_bike\\JC-201608-citibike-tripdata.csv',
 './data/city_bike\\JC-201609-citibike-tripdata.csv',
 './data/city_bike\\JC-20161-citibike-tripdata.csv',
 './data/city_bike\\JC-201610-citibike-tripdata.csv',
 './data/city_bike\\JC-201611-citibike-tripdata.csv',
 './data/city_bike\\JC-201612-citibike-tripdata.csv',
 './data/city_bike\\JC-20162-citibike-tripdata.csv',
 './data/city_bike\\JC-20163-citibike-tripdata.csv']

In [31]:
(bike_tables :=
 [pl.read_csv(p)
    .head(1)
     .with_columns(file = pl.lit(p.replace('\\','/').split('/')[-1]))
     .unpivot(index='file', variable_name='Column')
     .drop('value')
     .with_columns(pl.lit(1).alias('ones'))
     .pivot(index = 'Column', columns='file', values='ones')
  for p in bike_paths
  ]
)

C:\Users\kh6102sj\AppData\Local\Temp\ipykernel_9908\3292127372.py:8: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index = 'Column', columns='file', values='ones')


[shape: (15, 2)
 ┌───────────────────────┬─────────────────────────────────┐
 │ Column                ┆ JC-201604-citibike-tripdata.cs… │
 │ ---                   ┆ ---                             │
 │ str                   ┆ i32                             │
 ╞═══════════════════════╪═════════════════════════════════╡
 │ Trip Duration         ┆ 1                               │
 │ Start Time            ┆ 1                               │
 │ Stop Time             ┆ 1                               │
 │ Start Station ID      ┆ 1                               │
 │ Start Station Name    ┆ 1                               │
 │ …                     ┆ …                               │
 │ End Station Longitude ┆ 1                               │
 │ Bike ID               ┆ 1                               │
 │ User Type             ┆ 1                               │
 │ Birth Year            ┆ 1                               │
 │ Gender                ┆ 1                               │
 └──────

In [29]:
df1 = pl.DataFrame({'Column': ['a', 'b'],
                    'file1.csv':2*[1]})
df2 = pl.DataFrame({'Column': ['a', 'c'],
                    'file2.csv':2*[1]})

In [35]:
(df1.join(df2, 
          on='Column', 
          how = 'outer', 
          suffix = '_right',
          )
    .with_columns(Column = pl.coalesce('Column', 'Column_right'))
    .drop('Column_right')
)

join_next = lambda df1, df2: (df1.join(df2, 
                                       on='Column', 
                                       how = 'outer', 
                                       suffix = '_right')
                                 .with_columns(Column = pl.coalesce('Column', 'Column_right'))
                                 .drop('Column_right')
                             )

(bike_tables :=
 reduce(join_next, bike_tables)
 .fill_null(0)
 .sort('Column') # Should help find similar names/spellings/cases
)

(bike_tables :=
 bike_tables
 .with_columns(all = pl.reduce(mul, cs.starts_with('JC')),       # Multiple 1/0 columns <==> AND
               count = pl.reduce(add, cs.starts_with('JC')),
               )
)


C:\Users\kh6102sj\AppData\Local\Temp\ipykernel_9908\1463630377.py:1: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  (df1.join(df2,
C:\Users\kh6102sj\AppData\Local\Temp\ipykernel_9908\1463630377.py:10: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  join_next = lambda df1, df2: (df1.join(df2,


Column,JC-201604-citibike-tripdata.csv,JC-201605-citibike-tripdata.csv,JC-201606-citibike-tripdata.csv,JC-201607-citibike-tripdata.csv,JC-201608-citibike-tripdata.csv,JC-201609-citibike-tripdata.csv,JC-20161-citibike-tripdata.csv,JC-201610-citibike-tripdata.csv,JC-201611-citibike-tripdata.csv,JC-201612-citibike-tripdata.csv,JC-20162-citibike-tripdata.csv,JC-20163-citibike-tripdata.csv,all,count
str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""Bike ID""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""Birth Year""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""End Station ID""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""End Station Latitude""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""End Station Longitude""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Start Station Name""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""Start Time""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""Stop Time""",1,1,1,1,1,1,1,1,1,1,1,1,1,12


In [37]:
join_next = lambda df1, df2: (df1.join(df2, on='Column', how = 'full', suffix = '_right')
                                 .with_columns(Column = pl.coalesce('Column', 'Column_right'))
                                 .drop('Column_right')
                             )


(uber_column_summary :=
 pipe(glob('./data/city_bike/*.csv'),
     lambda L: [pl.read_csv(p)
                  .with_columns(file = pl.lit(p.replace('\\','/').split('/')[-1]))
                  .head(1) 
                  .unpivot(index='file', 
                           variable_name='Column')
                  .drop('value')
                  .with_columns(pl.lit(1).alias('ones'))
                  .pivot(index = 'Column', 
                         on='file', 
                         values='ones') 
                for p in L 
                ],
     lambda L: reduce(join_next, L).sort('Column').fill_null(0),
     lambda df: df.with_columns(all = pl.reduce(mul, cs.starts_with('JC')), 
                                count = pl.reduce(add, cs.starts_with('JC'))
                               ),
     )
)

Column,JC-201604-citibike-tripdata.csv,JC-201605-citibike-tripdata.csv,JC-201606-citibike-tripdata.csv,JC-201607-citibike-tripdata.csv,JC-201608-citibike-tripdata.csv,JC-201609-citibike-tripdata.csv,JC-20161-citibike-tripdata.csv,JC-201610-citibike-tripdata.csv,JC-201611-citibike-tripdata.csv,JC-201612-citibike-tripdata.csv,JC-20162-citibike-tripdata.csv,JC-20163-citibike-tripdata.csv,all,count
str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""Bike ID""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""Birth Year""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""End Station ID""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""End Station Latitude""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""End Station Longitude""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Start Station Name""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""Start Time""",1,1,1,1,1,1,1,1,1,1,1,1,1,12
"""Stop Time""",1,1,1,1,1,1,1,1,1,1,1,1,1,12


In [ ]:
(uber_column_summary
 .filter(pl.col('all') == 0)
)

Column,JC-201604-citibike-tripdata.csv,JC-201605-citibike-tripdata.csv,JC-201606-citibike-tripdata.csv,JC-201607-citibike-tripdata.csv,JC-201608-citibike-tripdata.csv,JC-201609-citibike-tripdata.csv,JC-20161-citibike-tripdata.csv,JC-201610-citibike-tripdata.csv,JC-201611-citibike-tripdata.csv,JC-201612-citibike-tripdata.csv,JC-20162-citibike-tripdata.csv,JC-20163-citibike-tripdata.csv,all,count
str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32


In [39]:
(uber_column_summary
 .filter(pl.col('count') != pl.col('count').max())
)

Column,JC-201604-citibike-tripdata.csv,JC-201605-citibike-tripdata.csv,JC-201606-citibike-tripdata.csv,JC-201607-citibike-tripdata.csv,JC-201608-citibike-tripdata.csv,JC-201609-citibike-tripdata.csv,JC-20161-citibike-tripdata.csv,JC-201610-citibike-tripdata.csv,JC-201611-citibike-tripdata.csv,JC-201612-citibike-tripdata.csv,JC-20162-citibike-tripdata.csv,JC-20163-citibike-tripdata.csv,all,count
str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32


<font color="orange">Your findings here.<font>
Well through processing the bike data we can see that there is NO bad files and no bad columns with differentiating data. Therefore this is a clean data set and ready to move forward with further processing 